# IT3091 - Member 2 - IT24100533 - Danthanarayana D.M.R


## Member 2 - Weather and location data

In [ ]:
!pip -q install openpyxl joblib

import os
import re
import json
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

REPO_PATH = Path("/content/IT3091---Machine-Learning-Assignment")
RAW_PATH = REPO_PATH / "Raw Datasets"
PREPROCESSED_PATH = REPO_PATH / "Preprocessed Datasets"

PREPROCESSED_PATH.mkdir(parents=True, exist_ok=True)

print("Raw data:", RAW_PATH)
print("Preprocessed data:", PREPROCESSED_PATH)

In [ ]:
WEATHER_FILE = RAW_PATH / "weatherData.csv"
LOCATION_FILE = RAW_PATH / "locationData.csv"

if not WEATHER_FILE.exists():
    matches = list(RAW_PATH.glob("*weatherData*.csv"))
    if matches:
        WEATHER_FILE = matches[0]

if not LOCATION_FILE.exists():
    matches = list(RAW_PATH.glob("*locationData*.csv"))
    if matches:
        LOCATION_FILE = matches[0]

if not WEATHER_FILE.exists() or not LOCATION_FILE.exists():
    raise FileNotFoundError("weatherData.csv or locationData.csv was not found in Raw Datasets.")

print("Weather file:", WEATHER_FILE.name)
print("Location file:", LOCATION_FILE.name)

In [ ]:
weather = pd.read_csv(WEATHER_FILE)
locations = pd.read_csv(LOCATION_FILE)

weather = weather.loc[:, ~weather.columns.astype(str).str.lower().isin(["index", "unnamed: 0"])]
locations = locations.loc[:, ~locations.columns.astype(str).str.lower().isin(["index", "unnamed: 0"])]

weather = weather.merge(
    locations[["location_id", "city_name"]],
    on="location_id",
    how="left",
    validate="many_to_one"
)

if weather["city_name"].isna().any():
    raise ValueError("Some location_id values did not match locationData.csv.")

weather["date"] = pd.to_datetime(weather["date"], errors="coerce")
weather = weather.dropna(subset=["date"]).copy()

print("Weather shape:", weather.shape)
display(weather.head())

In [ ]:
def season_and_year(dt):
    m, y = dt.month, dt.year

    if m in [4, 5, 6, 7, 8]:
        return pd.Series(["Yala", y])

    if m in [9, 10, 11, 12]:
        return pd.Series(["Maha", y])

    if m in [1, 2, 3]:
        return pd.Series(["Maha", y - 1])

    return pd.Series([pd.NA, pd.NA])

weather[["Season", "Year"]] = weather["date"].apply(season_and_year)
weather["Year"] = pd.to_numeric(weather["Year"], errors="coerce").astype("Int64")
weather = weather[weather["Year"].between(2012, 2023)].copy()

display(weather[["date", "Season", "Year"]].head())

In [ ]:
WEATHER_COLS = [
    "temperature_2m_mean (°C)",
    "precipitation_sum (mm)",
    "wind_speed_10m_max (km/h)",
    "shortwave_radiation_sum (MJ/m²)",
    "et0_fao_evapotranspiration (mm)"
]

for c in WEATHER_COLS:
    weather[c] = pd.to_numeric(weather[c], errors="coerce")

weather_daily_nat = (
    weather.groupby(["date", "Season", "Year"], as_index=False)[WEATHER_COLS]
           .mean()
)

weather_season = (
    weather_daily_nat.groupby(["Year", "Season"], as_index=False)
    .agg(
        SeasonMeanTemp_C=("temperature_2m_mean (°C)", "mean"),
        SeasonTotalPrecip_mm=("precipitation_sum (mm)", "sum"),
        SeasonMeanWind_kmh=("wind_speed_10m_max (km/h)", "mean"),
        SeasonMeanSolar_MJm2=("shortwave_radiation_sum (MJ/m²)", "mean"),
        SeasonTotalET0_mm=("et0_fao_evapotranspiration (mm)", "sum"),
        WeatherDays=("date", "nunique")
    )
)

display(weather_season.tail(10))

In [ ]:
weather_output = PREPROCESSED_PATH / "02_weather_preprocessed.csv"
weather_season.to_csv(weather_output, index=False)

print("Saved:", weather_output)
print("Rows:", len(weather_season))